# Systemy agentów

## 1. Wprowadzenie

**Agent** to system oparty na modelu językowym, który obserwuje otoczenie (np. dane, narzędzia, historię rozmowy), samodzielnie planuje kolejne kroki i wywołuje dostępne narzędzia, aby zrealizować cel użytkownika.

**Agentic RAG** to podejście, w którym agent nie tylko pobiera dokumenty (jak w klasycznym RAG), ale aktywnie decyduje, kiedy i jak je wyszukiwać, łączyć, filtrować oraz ewentualnie wykonywać dodatkowe kroki (np. dekompozycję zadania) przed wygenerowaniem odpowiedzi.

### Cel ćwiczenia
Zbudujemy agenta, który: na podstawie problemu zdrowotnego X, lokalizacji Y i innych preferencji uzytkownika zaproponuje przychodnię, korzystając z danych strukturalnych + opinii pacjentów.

## 2. Przygotowanie danych

In [ ]:
!pip install langgraph langchain-openai langchain-core
!pip install --quiet "transformers[torch]" accelerate sentencepiece safetensors
!pip install "langchain>=0.3" "langchain-community>=0.3" "langgraph>=1.0"
!pip install "faiss-cpu>=1.7" "sentence-transformers>=3.0"
!pip install "transformers[torch]>=4.45" "accelerate>=0.34" "safetensors>=0.4"
!pip install torch>=2.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 94.1 MB/s eta 0:00:00
/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: synt

In [ ]:
import pandas as pd

In [ ]:
clinics_df = pd.read_csv("clinics.csv")
clinics_df.head()

,clinic_id,name,city,county,voivodeship,address,diab,endo,cardio,pulm,nephro,rating,rating_count
0,1,"NIEPUBLICZNY ZAKŁAD OPIEKI ZDROWOTNEJ ""SOL-MED...",WARSZAWA,M. St. Warszawa,Mazowieckie,AL. 3 MAJA 2/35,0,0,0,1,0,4.2,15.0
1,2,"CENTRALNA WOJSKOWA PRZYCHODNIA LEKARSKA ""CEPEL...",WARSZAWA,M. St. Warszawa,Mazowieckie,KOSZYKOWA 78,0,0,1,0,0,2.5,581.0
2,3,"LECZNICA ""GOCŁAW"" MARIA KAMELA - PORADNIA LEKA...",WARSZAWA,M. St. Warszawa,Mazowieckie,AFRYKAŃSKA 7A,0,0,1,0,0,4.0,27.0
3,4,LUX MED ONKOLOGIA SPÓŁKA Z OGRANICZONĄ ODPOWIE...,WARSZAWA,M. St. Warszawa,Mazowieckie,"GEN. AUGUSTA EMILA FIELDORFA ""NILA"" 40",0,0,1,0,0,4.0,370.0
4,5,NIEPUBLICZNY ZAKŁAD OPIEKI ZDROWOTNEJ LECZNICA...,WARSZAWA,M. St. Warszawa,Mazowieckie,KLECZEWSKA 41,0,0,1,0,0,4.1,101.0


In [ ]:
reviews_df = pd.read_csv("clinic_reviews.csv")
reviews_df.head()

,clinic_id,review_id,rating,text
0,1,592b0f5a0e7e,1,"Brak możliwości zamówienia e-recepty online, a..."
1,1,89f48caae50f,2,"Cóż, dziwne miejsce w porównaniu do tych ,któr..."
2,1,36b5a47ad396,5,"Pani dr.Iwona Ģąsiewska bardzo miła,empatyczna..."
3,1,086485f820e2,5,Bardzo dobra przychodnia. Nie ma problemu z do...
4,1,aeb90042392b,4,w przypadku niewielkich dolegliwości i tylko t...


## 3. Integracja LangChain + LangGraph z lokalnym modelem


In [ ]:
import transformers
import torch

# model_name = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"
# opcjonalnie
model_name = "Qwen/Qwen2.5-3B-Instruct"
# model_name = "speakleash/Bielik-4.5B-v3.0-Instruct" # należy zarejestrować konto w Hugging Face, zatwierdzić warunki licencji na stronie Bielika i wgrać token dostępowy

prepared_prompt = "Szukam przychodni w Warszawie. Bolą mnie oczy. Jaka jest najbliższa przychodnia?"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    model_kwargs={"torch_dtype": torch.float16}, #float16 if not ampere/hopper gpu
    device_map="auto",
)

messages = [
    {"role": "system", "content": "You are a clinics advisor chatbot assistant!"},
    {"role": "user", "content": prepared_prompt},
]

outputs = pipeline(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

... (5 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


{'role': 'assistant', 'content': 'Przepraszam, ale nie mogę dostarczyć konkretnej informacji o najbliższej teraz przychodni ocznej w Warszawie bez więcej szczegółów. Czy mógłbyś podać dokładne lokalizację, gdzie jesteś? Takie informacje pomogłyby mi znaleźć odpowiednią i aktualną informację. Jeśli jesteś w Warszawie, możesz również skorzystać z aplikacji do map, które mogą podać dostępne teraz terapie oczne w pobliżu twojego położenia.'}


In [ ]:
import math
import json
from typing import Any, Dict, List, TypedDict

import pandas as pd
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.tools import StructuredTool, Tool
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langgraph.graph import END, StateGraph

# Reuse the transformers pipeline prepared above
llm = HuggingFacePipeline(pipeline=pipeline)

# Short LLM test via LangChain
quick_prompt = PromptTemplate.from_template("Podaj krótką poradę dla pacjenta z bólem głowy w Warszawie - gdzie ma się udać?")
quick_chain = LLMChain(llm=llm, prompt=quick_prompt)
_ = quick_chain.invoke({})
print("Test LangChain + lokalny model działa (odpowiedź przycięta)")


/tmp/ipython-input-3194986239.py:16: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipeline)
/tmp/ipython-input-3194986239.py:20: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  quick_chain = LLMChain(llm=llm, prompt=quick_prompt)


Test LangChain + lokalny model działa (odpowiedź przycięta)


## 4. Przygotowanie danych i prosty filtr kandydatów


In [ ]:
clinics_df["rating"] = pd.to_numeric(clinics_df["rating"], errors="coerce").fillna(0.0)
clinics_df["rating_count"] = pd.to_numeric(clinics_df["rating_count"], errors="coerce").fillna(0)

In [ ]:

# Mapowanie problem -> kolumny specjalizacji w danych strukturalnych
disease_to_specialty = {
    "cukrzyca": ["diab"],
    "diabet": ["diab"],
    "tarczy": ["endo"],
    "endo": ["endo"],
    "serc": ["cardio"],
    "kardio": ["cardio"],
    "płu": ["pulm"],
    "astm": ["pulm"],
    "nefro": ["nephro"],
    "nerk": ["nephro"],
}


def detect_specialties(problem: str) -> List[str]:
    text = (problem or "").lower()
    hits = []
    for keyword, cols in disease_to_specialty.items():
        if keyword in text:
            hits.extend(cols)
    return list(dict.fromkeys(hits))  # dedupe while keeping order


In [ ]:
def filter_clinics(problem: str, location: str = "", preferences: Dict[str, Any] | None = None, top_k: int = 30) -> List[Dict[str, Any]]:
    prefs = preferences or {}
    min_rating = float(prefs.get("min_rating", 0))
    min_count = int(prefs.get("min_rating_count", 0))

    specialties = detect_specialties(problem)
    df = clinics_df.copy()
    df = df[df["city"].str.upper() == "WARSZAWA"]

    if location:
        loc = location.upper()
        mask_loc = df["address"].str.upper().str.contains(loc) | df["name"].str.upper().str.contains(loc)
        df = df[mask_loc]

    if specialties:
        spec_mask = False
        for spec in specialties:
            spec_mask = spec_mask | (df[spec] == 1)
        df = df[spec_mask]

    df = df[(df["rating"] >= min_rating) & (df["rating_count"] >= min_count)]

    df = df.sort_values(by=["rating", "rating_count"], ascending=[False, False]).head(top_k)

    return [
        {
            "clinic_id": int(row.clinic_id),
            "name": row.name,
            "address": row.address,
            "rating": float(row.rating),
            "rating_count": int(row.rating_count),
            "specialties": specialties or "any",
        }
        for row in df.itertuples()
    ]


# Szybki test filtrowania
filter_clinics("problemy z sercem", "URSYNÓW", {"min_rating": 3.5, "min_rating_count": 50})[:3]


[{'clinic_id': 36,
  'name': 'SAMODZIELNY PUBLICZNY ZAKŁAD OPIEKI ZDROWOTNEJ WARSZAWA-URSYNÓW - PORADNIA PODSTAWOWEJ OPIEKI ZDROWOTNEJ',
  'address': 'KŁOBUCKA 14',
  'rating': 4.0,
  'rating_count': 55,
  'specialties': ['cardio']}]

## Opinie pacjentów: embeddingi i retriever

In [ ]:
reviews_df["rating"] = pd.to_numeric(reviews_df["rating"], errors="coerce").fillna(0.0)review_docs: List[Document] = [    Document(        page_content=str(row.text),        metadata={            "clinic_id": int(row.clinic_id),            "review_id": row.review_id,            "rating": float(row.rating),        },    )    for row in reviews_df.itertuples()]embedding_model_name = "sdadas/mmlw-retrieval-roberta-large"embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)vectorstore = FAISS.from_documents(review_docs, embeddings)reviews_retriever = vectorstore.as_retriever(search_kwargs={"k": 50})print(f"Załadowano {len(review_docs)} opinii, gotowy retriever RAG")

/tmp/ipython-input-4245684492.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

... (3 outputs left out)

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Załadowano 994 opinii, gotowy retriever RAG


## Narzędzia agenta (StructuredTool / Tool)

In [ ]:
def filter_clinics_tool(problem: str, location: str = "", preferences: Dict[str, Any] | None = None) -> List[Dict[str, Any]]:    return filter_clinics(problem, location, preferences or {}, top_k=30)def retrieve_reviews_tool(problem: str, clinic_ids: List[int], k_per_clinic: int = 3) -> List[Dict[str, Any]]:    """Select top opinions per clinic using similarity search and metadata filter."""    if not clinic_ids:        return []    retrieved = reviews_retriever.invoke(problem)    bucket: Dict[int, List[Document]] = {}    for doc in retrieved:        cid = int(doc.metadata.get("clinic_id", -1))        if cid not in clinic_ids:            continue        if len(bucket.get(cid, [])) >= k_per_clinic:            continue        bucket.setdefault(cid, []).append(doc)    flat: List[Dict[str, Any]] = []    for cid, docs in bucket.items():        for doc in docs:            flat.append(                {                    "clinic_id": cid,                    "review_id": doc.metadata.get("review_id"),                    "rating": float(doc.metadata.get("rating", 0)),                    "text": doc.page_content,                }            )    return flatdef rank_clinics(candidates: List[Dict[str, Any]], reviews: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:    if not candidates:        return []    review_score: Dict[int, float] = {}    review_count: Dict[int, int] = {}    for r in reviews:        cid = int(r["clinic_id"])        review_score[cid] = review_score.get(cid, 0.0) + float(r.get("rating", 0))        review_count[cid] = review_count.get(cid, 0) + 1    ranked = []    for c in candidates:        cid = c["clinic_id"]        avg_review = review_score.get(cid, 0.0) / review_count.get(cid, 1)        score = 0.6 * c["rating"] + 0.2 * math.log1p(c["rating_count"]) + 0.2 * avg_review        ranked.append({"score": score, **c})    ranked = sorted(ranked, key=lambda x: x["score"], reverse=True)[:top_k]    return rankedfilter_clinics_structured = StructuredTool.from_function(    name="filter_clinics_tool",    func=filter_clinics_tool,    description="Filtruje listę przychodni w Warszawie po specjalizacji, lokalizacji i ratingach.")retrieve_reviews = Tool(    name="retrieve_reviews_tool",    func=lambda query, clinic_ids: retrieve_reviews_tool(query, clinic_ids),    description="Zwraca trafne opinie pacjentów dla listy clinic_ids na podstawie zapytania.")print("Narzędzia zbudowane: filter_clinics_tool, retrieve_reviews_tool, rank_clinics")

Narzędzia zbudowane: filter_clinics_tool, retrieve_reviews_tool, rank_clinics


## Graf LangGraph: parse -> search -> reviews -> ranking -> answer

In [ ]:
class AgentState(TypedDict):    user_query: str    parsed_request: Dict[str, Any]    candidates: List[Dict[str, Any]]    retrieved_reviews: List[Dict[str, Any]]    ranked_clinics: List[Dict[str, Any]]    answer: strparse_prompt = PromptTemplate.from_template(    """    Z prośby użytkownika wyodrębnij:    - problem zdrowotny (problem)    - lokalizację (location, Warszawa jeśli brak)    - preferencje (preferences: min_rating, min_rating_count jeśli są)    Zwrot JSON bez komentarzy o kluczach: problem, location, preferences.    Prośba: {user_query}    """)parse_chain = LLMChain(llm=llm, prompt=parse_prompt)def parse_request_node(state: AgentState) -> Dict[str, Any]:    raw = parse_chain.invoke({"user_query": state["user_query"]})    try:        parsed = json.loads(raw["text"] if isinstance(raw, dict) else str(raw))    except Exception:        parsed = {            "problem": state.get("user_query", "problem zdrowotny"),            "location": "Warszawa",            "preferences": {},        }    return {"parsed_request": parsed}def structured_search_node(state: AgentState) -> Dict[str, Any]:    req = state.get("parsed_request", {})    candidates = filter_clinics(        req.get("problem", ""),        req.get("location", ""),        req.get("preferences", {}),        top_k=30,    )    return {"candidates": candidates}def reviews_rag_node(state: AgentState) -> Dict[str, Any]:    req = state.get("parsed_request", {})    clinic_ids = [c["clinic_id"] for c in state.get("candidates", [])][:20]    reviews = retrieve_reviews_tool(req.get("problem", ""), clinic_ids, k_per_clinic=3)    return {"retrieved_reviews": reviews}def ranking_node(state: AgentState) -> Dict[str, Any]:    ranked = rank_clinics(state.get("candidates", []), state.get("retrieved_reviews", []), top_k=5)    return {"ranked_clinics": ranked}answer_prompt = PromptTemplate.from_template(    """    Jesteś asystentem rekomendującym przychodnie POZ w Warszawie.    Użyj kandydatów i opinii, aby zaproponować maks 3 najlepsze opcje.    Problem: {problem}    Lokalizacja: {location}    Preferencje: {preferences}    Kandydaci (JSON): {candidates}    Opinie (JSON): {reviews}    Zwróć listę rekomendacji w języku polskim:    - nazwa przychodni + adres    - dlaczego pasuje (specjalizacja, ocena, liczba opinii)    - krótki cytat z opinii jeśli dostępny    """)answer_chain = LLMChain(llm=llm, prompt=answer_prompt)def answer_node(state: AgentState) -> Dict[str, Any]:    req = state.get("parsed_request", {})    text = answer_chain.invoke(        {            "problem": req.get("problem", ""),            "location": req.get("location", ""),            "preferences": req.get("preferences", {}),            "candidates": state.get("ranked_clinics", []),            "reviews": state.get("retrieved_reviews", []),        }    )    content = text["text"] if isinstance(text, dict) else str(text)    return {"answer": content}builder = StateGraph(AgentState)builder.add_node("parse_request", parse_request_node)builder.add_node("structured_search", structured_search_node)builder.add_node("reviews_rag", reviews_rag_node)builder.add_node("ranking", ranking_node)builder.add_node("answer", answer_node)builder.set_entry_point("parse_request")builder.add_edge("parse_request", "structured_search")builder.add_edge("structured_search", "reviews_rag")builder.add_edge("reviews_rag", "ranking")builder.add_edge("ranking", "answer")builder.add_edge("answer", END)app = builder.compile()print("Graf LangGraph skompilowany.")

Graf LangGraph skompilowany.


## Funkcja pomocnicza run_agent + przykładowe zapytania

In [ ]:
def run_agent(query: str) -> Dict[str, Any]:    state = app.invoke({"user_query": query})    return {        "answer": state.get("answer", ""),        "ranked": state.get("ranked_clinics", []),        "reviews": state.get("retrieved_reviews", []),    }examples = [    "Mieszkam na Ursynowie, mam cukrzycę i chcę dobrą ocenę oraz dużo opinii.",    "Szukam czegoś przy Grójeckiej, mam problemy z sercem i zależy mi na dobrych opiniach.",]for q in examples:    print("\n### Zapytanie:", q)    result = run_agent(q)    print(result["answer"][:800])    print("\nTop kandydaci (skrócone):")    for c in result["ranked"][:3]:        print({k: c[k] for k in ["clinic_id", "name", "rating", "rating_count", "score"]})


### Zapytanie: Mieszkam na Ursynowie, mam cukrzycę i chcę dobrą ocenę oraz dużo opinii.

    Jesteś asystentem rekomendującym przychodnie POZ w Warszawie.
    Użyj kandydatów i opinii, aby zaproponować maks 3 najlepsze opcje.

    Problem: Mieszkam na Ursynowie, mam cukrzycę i chcę dobrą ocenę oraz dużo opinii.
    Lokalizacja: Warszawa
    Preferencje: {}

    Kandydaci (JSON): [{'score': 4.283768121559319, 'clinic_id': 174, 'name': 'SAMODZIELNY ZESPÓŁ PUBLICZNYCH ZAKŁADÓW LECZNICTWA OTWARTEGO WARSZAWA WAWER - PORADNIA LEKARZA POZ', 'address': 'BEGONII 10', 'rating': 4.0, 'rating_count': 82, 'specialties': 'any'}, {'score': 3.7222534348808325, 'clinic_id': 134, 'name': 'PORADNIA LEKARZA POZ(CM LUX MED UL. GÓRCZEWSKA 124, 01-460 WARSZAWA; FILIA MR UL. WROCŁAWSKA 7A)', 'address': 'GÓRCZEWSKA 124', 'rating': 4.2, 'rating_count': 407, 'specialties': 'any'}, {'score': 3.614058382

Top kandydaci (skrócone):
{'clinic_id': 174, 'name': 'SAMODZIELNY ZESPÓŁ PUBLICZNYCH ZAKŁADÓW LECZNICTWA OTWA

## 9. Ograniczenia i możliwe rozszerzenia
- Brak geolokalizacji – filtr po adresie/dzielnicy jest tekstowy.
- Heurystyczne mapowanie problemów na specjalizacje; warto rozbudować słownik lub użyć klasyfikatora.
- Ranking korzysta z prostych wag; można dodać sentyment opinii lub model oceny jakości.
- Opinie są indeksowane jedynie dla wyszukiwania semantycznego; brak filtrowania po świeżości opinii.
- Można dodać kolejkowanie narzędzi (np. sprawdzanie dostępności terminów) i lepszą obsługę błędów parsowania JSON w węźle `parse_request`.



## Węzeł filtrowania jakościDodatkowy węzeł grafu odsiewający kandydatów poniżej progu oceny i liczby opinii, z rozluźnianiem kryteriów, gdy zostałoby ich mniej niż trzech.

In [ ]:
from typing import Dict, Any

def quality_filter_node(state: AgentState) -> Dict[str, Any]:
    """
    Filtruje kandydatów według kryteriów jakości.
    Jeśli zostaje < 3 kandydatów, rozluźnia kryteria.
    """
    # pobierz kandydatów ze stanu
    candidates = state.get("candidates", [])
    original_count = len(candidates)
    # TODO
    # KROK 1: Pierwsze filtrowanie (rygorystyczne)
    min_rating_strict = 3.5
    min_count_strict = 10

    filtered = [
        c for c in candidates
        if c.get('rating', 0) >= min_rating_strict and c.get('rating_count', 0) >= min_count_strict
    ]

    # KROK 2: Jeśli za mało, rozluźnij kryteria
    if len(filtered) < 3:
        min_rating_relaxed = 3.0

        filtered = [
            c for c in candidates
            if c.get('rating', 0) >= min_rating_relaxed
        ]

    # KROK 3: Logowanie końcowe
    rejected_count = original_count - len(filtered)
    print(f"Zaakceptowano: {len(filtered)} przychodni")
    print(f"Odrzucono: {rejected_count} (niski rating lub za mało opinii)")

    # Jeśli nadal 0 kandydatów, zwróć oryginalne (fallback)
    if len(filtered) == 0:
        print("  UWAGA: Żaden kandydat nie spełnia kryteriów - zwracam wszystkich")
        filtered = candidates

    return {"candidates": filtered}

In [ ]:
from langgraph.graph import StateGraph, END

# Przebuduj graf
builder = StateGraph(AgentState)

# Dodaj wszystkie węzły
builder.add_node("parse_request", parse_request_node)
builder.add_node("structured_search", structured_search_node)
builder.add_node("quality_filter", quality_filter_node)
builder.add_node("reviews_rag", reviews_rag_node)
builder.add_node("ranking", ranking_node)
builder.add_node("answer", answer_node)

builder.set_entry_point("parse_request")

# Definiuj przepływ
builder.add_edge("parse_request", "structured_search")
builder.add_edge("structured_search", "quality_filter")
builder.add_edge("quality_filter", "reviews_rag")
builder.add_edge("reviews_rag", "ranking")
builder.add_edge("ranking", "answer")
builder.add_edge("answer", END)

# Skompiluj NOWY graf
app = builder.compile()

In [ ]:
# przykład

query1 = "Szukam przychodni na Ursynowie z diabetologiem"
result1 = run_agent(query1)

print(f"\nZnaleziono: {len(result1['ranked'])} przychodni")
for i, clinic in enumerate(result1['ranked'][:3], 1):
    print(f"{i}. {clinic['name'][:50]} - Rating: {clinic['rating']:.1f}/5.0")

Zaakceptowano: 14 przychodni
Odrzucono: 16 (niski rating lub za mało opinii)

Znaleziono: 5 przychodni
1. CENTERMED WARSZAWA SP Z O. O - PORADNIA LEKARZA PO - Rating: 4.1/5.0
2. PORADNIA LEKARZA POZ(CM LUX MED UL. GÓRCZEWSKA 124 - Rating: 4.2/5.0
3. ZAKŁAD OPIEKI ZDROWOTNEJ WARSZAWA BEMOWO-WŁOCHY -  - Rating: 4.3/5.0
